# Exercise 4: Mutual intelligibility and NLP

*The Turkic World · version 0.1.0*

> The Kardeş-NLU paper (Şenel et al., EACL 2024) studies transfer to five Turkic languages with Turkish as a high-resource relative. Before reading it, rank the five languages (Azerbaijani, Uzbek, Kyrgyz, Kazakh, Uyghur) by how much you expect them to benefit from Turkish, based on genealogy, script, and resources. Then compare your ranking with the paper's results. Which factors are confounded in this comparison, and what controlled experiment would distinguish them?

**Setup.** `pip install "turkicnlp==0.5.0" requests pandas`. Cells that query websites need internet access; if a request fails, the cell records the failure instead of a number. An unavailable value is not zero.

In [ ]:
import datetime
import requests
import pandas as pd

ACCESSED = datetime.date.today().isoformat()        # record the access date with every observation
HEADERS = {"User-Agent": "turkic-nlp-book-exercises/0.1 (educational use)"}

def get_json(url, **params):
    """Return parsed JSON, or {'_error': ...} if the request fails (never a fake value)."""
    try:
        r = requests.get(url, params=params or None, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        return {"_error": f"{type(exc).__name__}: {exc}"}

def get_text(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.text
    except Exception as exc:
        return None

def github_org_repos(org):
    """All public repositories of a GitHub organisation (paged listing; avoids the search API's low rate limit)."""
    repos, page = [], 1
    while True:
        d = get_json(f"https://api.github.com/orgs/{org}/repos", per_page=100, page=page)
        if not isinstance(d, list):
            return repos, d.get("_error")
        repos += d
        if len(d) < 100:
            return repos, None
        page += 1

In [ ]:
# Step 1 — your prediction BEFORE looking at the results (most benefit first). Edit this list.
MY_RANKING = ["Azerbaijani", "Uzbek", "Kyrgyz", "Kazakh", "Uyghur"]

# Step 2 — the three factors for each language
# branch: The Turkic World branch table; script: script of the Kardeş-NLU test data (Şenel et al. 2024);
# resources: English–X training pairs in the TIL release (tables/tbl-til-data/data.csv in the book repository)
factors = pd.DataFrame([
    {"language": "Azerbaijani", "code": "az", "branch": "SW (Oghuz), same as Turkish", "test script": "Latin",    "TIL en-X pairs": 548_901},
    {"language": "Uzbek",       "code": "uz", "branch": "SE (Karluk)",                 "test script": "Latin",    "TIL en-X pairs": 529_574},
    {"language": "Kyrgyz",      "code": "ky", "branch": "NW (Kipchak)",                "test script": "Cyrillic", "TIL en-X pairs": 312_644},
    {"language": "Kazakh",      "code": "kk", "branch": "NW (Kipchak)",                "test script": "Cyrillic", "TIL en-X pairs": 564_760},
    {"language": "Uyghur",      "code": "ug", "branch": "SE (Karluk)",                 "test script": "Cyrillic", "TIL en-X pairs": 96_898},
])
factors

In [ ]:
# Step 3 — the paper's results: NLI accuracy (%), XLM-R Base, zero-shot (Şenel et al. 2024, Table 3).
# "EN" = fine-tuned on English only; "EN+TR, BJLM" = the paper's headline Turkish-assisted setting.
# Transcribed from the paper; check against the PDF before reusing the numbers.
nli = pd.DataFrame([
    {"language": "Azerbaijani", "EN": 76.5, "EN+TR, BJLM": 79.3},
    {"language": "Kazakh",      "EN": 73.8, "EN+TR, BJLM": 76.8},
    {"language": "Kyrgyz",      "EN": 70.4, "EN+TR, BJLM": 75.5},
    {"language": "Uzbek",       "EN": 70.7, "EN+TR, BJLM": 75.7},
    {"language": "Uyghur",      "EN": 42.2, "EN+TR, BJLM": 59.4},
])
nli["gain"] = (nli["EN+TR, BJLM"] - nli["EN"]).round(1)
result = nli.merge(factors, on="language").sort_values("gain", ascending=False).reset_index(drop=True)
result["observed rank"] = range(1, len(result) + 1)
result["my rank"] = result["language"].map({l: i + 1 for i, l in enumerate(MY_RANKING)})
result[["language", "EN", "EN+TR, BJLM", "gain", "observed rank", "my rank", "branch", "test script", "TIL en-X pairs"]]

In [ ]:
# How well does each single factor "explain" the gains? (5 languages: descriptive only)
print("Spearman correlation of gain with TIL resources:",
      round(result["gain"].corr(result["TIL en-X pairs"], method="spearman"), 2))
print("Spearman correlation of gain with my ranking:",
      round(result["gain"].corr(-result["my rank"], method="spearman"), 2))

## Worked solution

**A reasonable prediction** ranked Azerbaijani first (same Oghuz branch and Latin script as Turkish) and Uyghur last (Karluk branch, least data). The paper finds almost the opposite: on NLI, **Uyghur gains most by a wide margin** (42.2 → 59.4), Kyrgyz and Uzbek gain about 5 points, and **Azerbaijani and Kazakh gain least** (the paper notes that these two are the best-resourced of the five).

**Why the factors are confounded.** Uyghur is at once the language with the least data, the one with the lowest baseline (more room to improve), and the one with a script mismatch: the paper notes that most of XLM-R's Uyghur pretraining text is in Arabic script while the test data is Cyrillic, so its inputs are split into many more tokens. Azerbaijani is both the closest relative of Turkish and among the best-resourced, so a small gain cannot be attributed to either property alone. With five languages, every factor is entangled with the others; the correlations above are descriptive only.

**A controlled experiment.** Hold the language fixed and vary one factor at a time:

- *Script:* evaluate the same Uyghur test items in Arabic and in Cyrillic script, with and without Turkish intermediate training.
- *Relatedness:* for one target (for example Kyrgyz), compare Turkish with an unrelated high-resource donor given the same amount of data.
- *Resources:* subsample a well-resourced target's pretraining or fine-tuning data to the size of a low-resource one and repeat the comparison.

Report paired differences on the same test items, with several random seeds.